# Modellvergleich: Baseline → LightGBM v1 → v2 → XGBoost

Dieses Notebook beantwortet drei Fragen:

1. **Was bringt das Kaskadenfeature?** — LightGBM v1 vs. v2 (isolierter Feature-Effekt)
2. **Ist LightGBM der beste Algorithmus?** — v2 vs. XGBoost (gleiche Features, anderer Algorithmus)
3. **Wo bleibt Fehler übrig?** — Segmentanalyse aller Modelle (Stunde, Linie, Wetter)

**Erwartung aus der Analyse:** `prev_trip_delay` (r ≥ 0.85 im Kaskadeneffekt) sollte signifikant helfen. Algorithmus-Unterschied zwischen LightGBM und XGBoost sollte gering sein — beide sind Gradient Boosting mit ähnlichen Inductive Biases.

**Voraussetzung:** `06_prediction_4-model_v2.ipynb` vollständig ausgeführt — `lgbm_v2.txt`, `test_predictions_v2.parquet` und `test_final_v2.parquet` müssen vorhanden sein.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import polars as pl
import lightgbm as lgb
import xgboost as xgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json as _json
from pathlib import Path

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("06_prediction_5-comparison")

processed_dir = Path(str(TRAIN)).parent
models_dir    = processed_dir.parent / "models"

# Paths
train_v2_path    = str(processed_dir / "train_final_v2.parquet")
test_v2_path     = str(processed_dir / "test_final_v2.parquet")
pred_v1_path     = str(processed_dir / "test_predictions.parquet")
pred_v2_path     = str(processed_dir / "test_predictions_v2.parquet")

# Metadaten laden
with open(models_dir / "lgbm_v1_meta.json") as f:
    v1_meta = _json.load(f)
with open(models_dir / "lgbm_v2_meta.json") as f:
    v2_meta = _json.load(f)

BASELINE_MAE = 50.0
TARGET       = "arrival_delay"
print("Setup abgeschlossen.")

## Daten laden

Predictions aller bisherigen Modelle laden — kein erneutes Training nötig.

In [ ]:
# v1 Predictions (aus 06_prediction_2-model)
pred_v1 = pl.read_parquet(pred_v1_path)
y_test  = pred_v1["actual"].to_numpy()

# v2 Predictions (aus 06_prediction_4-model_v2)
pred_v2 = pl.read_parquet(pred_v2_path)

print(f"Test-Set:      {len(y_test):,} Beobachtungen")
print(f"v1 Prediction: {pred_v1.columns}")
print(f"v2 Prediction: {pred_v2.columns}")

In [ ]:
# Feature-Setup für XGBoost (gleiche v2-Features)
FEATURES_V2 = v2_meta["features"]
CAT_COLS_V2 = v2_meta["cat_cols"]

def to_xgb_df(pl_df: pl.DataFrame, features: list, cat_cols: list) -> pd.DataFrame:
    """Polars → Pandas mit pandas Categorical für XGBoost enable_categorical."""
    pdf = pl_df.select(features).to_pandas()
    for col in cat_cols:
        if col in pdf.columns:
            pdf[col] = pdf[col].astype("category")
    return pdf

# Validation-Split (identisch v1/v2 für faires Early-Stopping)
print("Lade train_final_v2 ...")
train_v2_pl = pl.read_parquet(train_v2_path)

val_mask   = (
    (train_v2_pl["operating_date"].dt.year() == 2024)
    & (train_v2_pl["operating_date"].dt.month() >= 7)
)
train_sub = train_v2_pl.filter(~val_mask)
val_sub   = train_v2_pl.filter(val_mask)

print(f"Train: {len(train_sub):,}  ·  Val: {len(val_sub):,}")

print("Konvertiere zu Pandas ...")
X_train = to_xgb_df(train_sub, FEATURES_V2, CAT_COLS_V2)
y_train = train_sub[TARGET].to_numpy()
X_val   = to_xgb_df(val_sub,   FEATURES_V2, CAT_COLS_V2)
y_val   = val_sub[TARGET].to_numpy()
print("Fertig.")

## Vergleichsmodell: XGBoost

XGBoost mit `enable_categorical=True` (ab Version 2.0) — vergleichbarer nativer Categorical-Support wie LightGBM. Gleiche Feature-Set wie v2 für sauberen Algorithmen-Vergleich.

**Kernunterschiede LightGBM vs. XGBoost:**
| | LightGBM | XGBoost |
|:---|:---|:---|
| Baumwachstum | Leaf-wise (bestes Blatt) | Level-wise (vollständige Ebene) |
| Geschwindigkeit | Schneller (GOSS/EFB) | Langsamer auf grossen Datasets |
| Categorical | Nativ (optimal grouping) | Nativ (ab 2.0, partitionsbasiert) |
| Regularisierung | L1/L2 + min_gain | L1/L2 + gamma |

In [ ]:
import time

xgb_model = xgb.XGBRegressor(
    n_estimators        = 1000,
    learning_rate       = 0.05,
    max_depth           = 6,          # entspricht ungefähr num_leaves=63 in LGBM
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    min_child_weight    = 50,
    objective           = "reg:absoluteerror",   # MAE-Optimierung
    eval_metric         = "mae",
    early_stopping_rounds = 50,
    enable_categorical  = True,        # XGBoost 2.0+ — native Categorical
    tree_method         = "hist",      # schnell + categorical-kompatibel
    device              = "cpu",
    n_jobs              = -1,
    random_state        = 42,
    verbosity           = 0,
)

print("XGBoost Training startet ...")
t0 = time.time()
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)
train_time_xgb = time.time() - t0

xgb_best_iter  = xgb_model.best_iteration
xgb_val_mae    = xgb_model.best_score
print(f"\nBeste Iteration: {xgb_best_iter}")
print(f"Bestes Val-MAE:  {xgb_val_mae:.2f} s")
print(f"Trainingszeit:   {train_time_xgb:.0f} s")

In [ ]:
# XGBoost Test-Predictions
test_v2_pl   = pl.read_parquet(test_v2_path)
X_test_xgb   = to_xgb_df(test_v2_pl, FEATURES_V2, CAT_COLS_V2)
y_test_xgb   = test_v2_pl[TARGET].to_numpy()

test_pred_xgb = xgb_model.predict(X_test_xgb)

xgb_test_mae  = np.abs(y_test_xgb - test_pred_xgb).mean()
xgb_test_mbe  = (test_pred_xgb - y_test_xgb).mean()
xgb_test_rmse = np.sqrt(((y_test_xgb - test_pred_xgb) ** 2).mean())
xgb_test_otp  = (np.abs(y_test_xgb - test_pred_xgb) <= 60).mean()

print(f"XGBoost Test — MAE: {xgb_test_mae:.1f} s  ·  MBE: {xgb_test_mbe:+.1f} s")

# Export
pred_xgb_df = pl.DataFrame({
    "actual":       y_test_xgb,
    "predicted_xgb": test_pred_xgb.astype("float32"),
    "line_name":    test_v2_pl["line_name"],
    "stop_name":    test_v2_pl["stop_name"],
    "hour":         test_v2_pl["hour"],
    "month":        test_v2_pl["month"],
    "has_rain":     test_v2_pl["has_rain"],
    "has_snow":     test_v2_pl["has_snow"],
    "has_event":    test_v2_pl["has_event"],
})
pred_xgb_path = processed_dir / "test_predictions_xgb.parquet"
pred_xgb_df.write_parquet(pred_xgb_path)
print(f"XGBoost Predictions gespeichert: {pred_xgb_path}")

# XGBoost Modell
xgb_model_path = models_dir / "xgboost_v1.json"
xgb_model.save_model(str(xgb_model_path))
print(f"XGBoost Modell gespeichert: {xgb_model_path}")

## Metriken-Vergleich: Alle Modelle

Vollständige Übersicht — Baseline bis XGBoost.

In [ ]:
# Metriken-Tabelle aufbauen
y_v1  = pred_v1["actual"].to_numpy()
p_v1  = pred_v1["predicted"].to_numpy()
p_v2  = pred_v2["predicted_v2"].to_numpy()
p_v2c = pred_v2["predicted_v2_cal"].to_numpy()

def metrics(actual, predicted, label):
    mae  = np.abs(actual - predicted).mean()
    rmse = np.sqrt(((actual - predicted) ** 2).mean())
    mbe  = (predicted - actual).mean()
    otp  = (np.abs(actual - predicted) <= 60).mean()
    return {"Modell": label, "MAE (s)": round(mae, 1), "RMSE (s)": round(rmse, 1),
            "MBE (s)": round(mbe, 1), "OTP ±60s": f"{otp:.1%}"}

rows = [
    {"Modell": "Baseline (Stop Mean)",   "MAE (s)": 50.0,  "RMSE (s)": "—",  "MBE (s)": "—",   "OTP ±60s": "—"},
    metrics(y_v1,       p_v1,         "LightGBM v1 (ohne Kaskade)"),
    metrics(y_test_xgb, p_v2,         "LightGBM v2 (+ Kaskade)"),
    metrics(y_test_xgb, p_v2c,        "LightGBM v2 kalibriert"),
    metrics(y_test_xgb, test_pred_xgb, "XGBoost (+ Kaskade)"),
]
metrics_df = pd.DataFrame(rows)
show_df(metrics_df)

In [ ]:
# Visualisierung: MAE-Vergleich als Balkendiagramm
labels = ["Baseline", "LGBM v1", "LGBM v2", "LGBM v2\nkalib.", "XGBoost"]
maes   = [
    BASELINE_MAE,
    np.abs(y_v1 - p_v1).mean(),
    np.abs(y_test_xgb - p_v2).mean(),
    np.abs(y_test_xgb - p_v2c).mean(),
    xgb_test_mae,
]
colors = ["#8c8c8c", "#4c72b0", "#55a868", "#25ac82", "#dd8452"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, maes, color=colors, alpha=0.85, edgecolor="white", linewidth=0.5)

for bar, val in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f"{val:.1f} s", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylim(0, max(maes) * 1.15)
ax.set_ylabel("Test MAE (s)")
ax.set_title("Modellvergleich — Test MAE (2025 Test-Set)\nKleiner = besser", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Was bringt das Kaskadenfeature?

Isolierter Effekt von `prev_trip_delay` + `stop_sequence_pct`: LightGBM v1 vs. v2, identische Hyperparameter.

In [ ]:
# Feature Importance Vergleich: v1 vs. v2
lgb_v1 = lgb.Booster(model_file=str(models_dir / "lgbm_v1.txt"))
lgb_v2 = lgb.Booster(model_file=str(models_dir / "lgbm_v2.txt"))

imp_v1 = pd.DataFrame({
    "feature": lgb_v1.feature_name(),
    "gain_v1": lgb_v1.feature_importance(importance_type="gain"),
})
imp_v2 = pd.DataFrame({
    "feature": lgb_v2.feature_name(),
    "gain_v2": lgb_v2.feature_importance(importance_type="gain"),
})

# Normalisieren (0–100) für fairen Vergleich trotz unterschiedlicher Iterationsanzahl
imp_v1["gain_v1"] = imp_v1["gain_v1"] / imp_v1["gain_v1"].sum() * 100
imp_v2["gain_v2"] = imp_v2["gain_v2"] / imp_v2["gain_v2"].sum() * 100

imp_cmp = imp_v2.merge(imp_v1, on="feature", how="left").fillna(0)
imp_cmp = imp_cmp.sort_values("gain_v2", ascending=False).head(20)

# Neue Features hervorheben
new_feats = v2_meta["new_features"]
print(f"Neue Features in v2: {new_feats}")
print()

fig, ax = plt.subplots(figsize=(11, 7))
y_pos = range(len(imp_cmp))

ax.barh([i + 0.2 for i in y_pos], imp_cmp["gain_v2"],
        height=0.4, label="LightGBM v2", color="#55a868", alpha=0.85)
ax.barh([i - 0.2 for i in y_pos], imp_cmp["gain_v1"],
        height=0.4, label="LightGBM v1", color="#4c72b0", alpha=0.85)

ax.set_yticks(list(y_pos))
ax.set_yticklabels([
    f"★ {f}" if f in new_feats else f
    for f in imp_cmp["feature"]
], fontsize=9)
ax.set_xlabel("Normalisierter Gain (%)")
ax.set_title("Feature Importance — v1 vs. v2 (★ = neue Features)\nnormalisiert auf 100%",
             fontsize=11)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

show_df(imp_cmp.reset_index(drop=True))

## Fehler nach Segment: Alle Modelle

Wo ist welches Modell gut, wo bleibt Fehler übrig?

In [ ]:
# Gemeinsames Error-DataFrame aufbauen
# p_v1 kommt aus test_predictions.parquet (test_final), p_v2/xgb aus test_final_v2
# Beide Test-Sets sind 2025 — sollten identisch viele Zeilen haben
assert len(p_v1) == len(y_test_xgb), (
    f"Längen stimmen nicht überein: p_v1={len(p_v1)}, y_test_xgb={len(y_test_xgb)}"
)

base_df = test_v2_pl.to_pandas()[["line_name", "hour", "has_rain", "has_heavy_rain", "has_snow"]].copy()
base_df["actual"]  = y_test_xgb
base_df["ae_v1"]   = np.abs(y_test_xgb - p_v1)
base_df["ae_v2"]   = np.abs(y_test_xgb - p_v2)
base_df["ae_xgb"]  = np.abs(y_test_xgb - test_pred_xgb)

model_cols   = {"LGBM v1": "ae_v1", "LGBM v2": "ae_v2", "XGBoost": "ae_xgb"}
model_colors = {"LGBM v1": "#4c72b0", "LGBM v2": "#55a868", "XGBoost": "#dd8452"}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. MAE nach Stunde
ax = axes[0, 0]
for name, col in model_cols.items():
    mae_h = base_df.groupby("hour")[col].mean()
    ax.plot(mae_h.index, mae_h.values, label=name, color=model_colors[name], lw=2)
ax.set_title("MAE nach Tageszeit"); ax.set_xlabel("Stunde"); ax.set_ylabel("MAE (s)")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)

# 2. MAE nach Linie
ax = axes[0, 1]
mae_line_all = (
    base_df.groupby("line_name")[["ae_v1", "ae_v2", "ae_xgb"]].mean()
    .sort_values("ae_v2", ascending=False)
)
x = range(len(mae_line_all))
w = 0.28
for i, (name, col) in enumerate(model_cols.items()):
    ax.bar([xi + (i - 1) * w for xi in x], mae_line_all[col],
           width=w, label=name, color=model_colors[name], alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(mae_line_all.index.astype(str), rotation=45)
ax.set_title("MAE nach Linie"); ax.set_ylabel("MAE (s)")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)

# 3. MAE nach Wetter
ax = axes[1, 0]
weather_map = {
    "Normal":     (~base_df["has_rain"]) & (~base_df["has_snow"]),
    "Regen":      base_df["has_rain"] & ~base_df["has_snow"],
    "Starkregen": base_df["has_heavy_rain"],
    "Schnee":     base_df["has_snow"],
}
x_w = range(len(weather_map))
for i, (name, col) in enumerate(model_cols.items()):
    vals = [base_df.loc[mask, col].mean() for mask in weather_map.values()]
    ax.bar([xi + (i - 1) * w for xi in x_w], vals,
           width=w, label=name, color=model_colors[name], alpha=0.85)
ax.set_xticks(list(x_w))
ax.set_xticklabels(list(weather_map.keys()))
ax.set_title("MAE nach Wetter"); ax.set_ylabel("MAE (s)")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)

# 4. Residual-Verteilung
ax = axes[1, 1]
bins = np.linspace(-150, 150, 60)
sample_idx = np.random.default_rng(42).choice(len(y_test_xgb), 50_000, replace=False)
residuals = {
    "LGBM v1":  p_v1          - y_test_xgb,
    "LGBM v2":  p_v2          - y_test_xgb,
    "XGBoost":  test_pred_xgb - y_test_xgb,
}
for name, res in residuals.items():
    ax.hist(res[sample_idx], bins=bins, alpha=0.5, label=name,
            color=model_colors[name], density=True)
ax.axvline(0, color="black", lw=1.5, ls="--")
ax.set_xlabel("Residual (predicted − actual, s)")
ax.set_title("Residual-Verteilung (Stichprobe n=50k)")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Fehleranalyse — Alle Modelle im Vergleich", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Fazit und Empfehlung

In [ ]:
# Zusammenfassende Tabelle
v1_mae  = np.abs(y_v1  - p_v1).mean()  if len(p_v1) == len(y_v1) else float("nan")
v2_mae  = np.abs(y_test_xgb - p_v2).mean()
v2c_mae = np.abs(y_test_xgb - p_v2c).mean()
xgb_mae = xgb_test_mae

summary = pd.DataFrame([
    {"Modell": "Baseline (Stop Mean)",   "Test MAE": 50.0, "Δ Baseline": 0.0,
     "Kaskadenfeature": "—",  "Empfehlung": "Referenz"},
    {"Modell": "LightGBM v1",            "Test MAE": round(v1_mae, 1),
     "Δ Baseline": round(50.0 - v1_mae, 1),
     "Kaskadenfeature": "Nein",  "Empfehlung": "Benchmark"},
    {"Modell": "LightGBM v2",            "Test MAE": round(v2_mae, 1),
     "Δ Baseline": round(50.0 - v2_mae, 1),
     "Kaskadenfeature": "Ja",   "Empfehlung": "Bevorzugt (schnell)"},
    {"Modell": "LightGBM v2 kalibriert", "Test MAE": round(v2c_mae, 1),
     "Δ Baseline": round(50.0 - v2c_mae, 1),
     "Kaskadenfeature": "Ja",   "Empfehlung": "Bevorzugt (Echtzeit)"},
    {"Modell": "XGBoost",                "Test MAE": round(xgb_mae, 1),
     "Δ Baseline": round(50.0 - xgb_mae, 1),
     "Kaskadenfeature": "Ja",   "Empfehlung": "Robustheits-Check"},
])
show_df(summary)

### Erkenntnisse

**1. Kaskadenfeature**  
Der Effekt von `prev_trip_delay` zeigt, wie stark die analytische Erkenntnis (Pearson r ≥ 0.85) im Modell nutzbar ist. Wenn das Feature oben in der Feature Importance steht, bestätigt das: die Kaskade ist kein statistisches Artefakt, sondern ein echtes, lernbares Signal.

**2. LightGBM vs. XGBoost**  
Beide Algorithmen arbeiten auf dem gleichen Feature-Set — ein grosser Unterschied im MAE wäre ein Zeichen, dass einer der beiden besser zur Datenstruktur passt (oder dass Hyperparameter-Tuning noch Luft lässt). Ein kleiner Unterschied bestätigt: das Signal steckt in den Daten, nicht im Algorithmus.

**3. Empfehlung**  
* **Operativer Einsatz (Echtzeit):** LightGBM v2 kalibriert — geringster Bias, schnell, native Categorical-Unterstützung  
* **Portfolio-Darstellung:** LightGBM v2 + XGBoost als Robustheits-Check — zeigt, dass das Ergebnis algorithmus-unabhängig ist  
* **Nächster Schritt:** Optuna-Tuning auf LightGBM v2 (→ `06_prediction_6-tuning.ipynb`) könnte weitere 1–3 s MAE herausholen